In [1]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import shap
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import r2_score

# ==========================================
# 1. 数据集定义 (缺失值自动填补 + 断层检测)
# ==========================================
class TimeAwareMultiStationDataset(Dataset):
    # 修改1: 默认 target_col 改为夜间分配法 GPP_NT_VUT_REF
    def __init__(self, filepaths, seq_len=24, target_col='GPP_NT_VUT_REF', time_col='date',
                 forcing_cols=None, state_cols=None,
                 static_cols=['Lat', 'Long'], lc_col='Veg_ID',
                 feat_min_f=None, feat_max_f=None, feat_min_s=None, feat_max_s=None,
                 static_min=None, static_max=None, split_type='train'):

        self.seq_len = seq_len
        self.samples = []

        self.station_forcing = []
        self.station_state = []
        self.station_time_features = []
        self.station_targets = []
        self.station_dates = []

        self.station_static = []
        self.station_lc = []

        for filepath in filepaths:
            data = pd.read_csv(filepath)
            if time_col not in data.columns:
                print(f"⚠️ 在文件 {filepath} 中找不到时间列 '{time_col}'，跳过。")
                continue

            # 异常值清洗与插值填补 NaN
            data = data.replace([-9999, -9999.0, -999], np.nan)
            cols_to_clean = forcing_cols + state_cols + [target_col]
            cols_exist = [c for c in cols_to_clean if c in data.columns]

            data[cols_exist] = data[cols_exist].interpolate(method='linear', limit_direction='both')
            data = data.dropna(subset=cols_exist).reset_index(drop=True)

            data[time_col] = pd.to_datetime(data[time_col])
            data = data.sort_values(by=time_col).reset_index(drop=True)

            if len(data) < seq_len:
                continue

            dates = data[time_col]
            self.station_dates.append(dates.values)

            hours = dates.dt.hour.values
            days = dates.dt.dayofyear.values
            time_feats = np.column_stack([
                np.sin(2 * np.pi * hours / 24.0), np.cos(2 * np.pi * hours / 24.0),
                np.sin(2 * np.pi * days / 365.25), np.cos(2 * np.pi * days / 365.25)
            ])

            forcing_data = data[forcing_cols].values
            state_data = data[state_cols].values
            static_data = data[static_cols].values
            lc_data = data[lc_col].values.astype(np.int64)

            if target_col in data.columns:
                targets = data[target_col].values
            else:
                targets = data.iloc[:, -1].values

            self.station_forcing.append(forcing_data)
            self.station_state.append(state_data)
            self.station_time_features.append(time_feats)
            self.station_targets.append(targets)
            self.station_static.append(static_data)
            self.station_lc.append(lc_data)

        if not self.station_forcing:
            raise ValueError(f"加载 {split_type} 数据失败，可能所有数据均被清洗掉或长度不足。")

        all_forcing_concat = np.vstack(self.station_forcing)
        all_state_concat = np.vstack(self.station_state)
        all_static_concat = np.vstack(self.station_static)

        self.feat_min_f = np.min(all_forcing_concat, axis=0) if feat_min_f is None else feat_min_f
        self.feat_max_f = np.max(all_forcing_concat, axis=0) if feat_max_f is None else feat_max_f
        self.feat_min_s = np.min(all_state_concat, axis=0) if feat_min_s is None else feat_min_s
        self.feat_max_s = np.max(all_state_concat, axis=0) if feat_max_s is None else feat_max_s

        self.static_min = np.min(all_static_concat, axis=0) if static_min is None else static_min
        self.static_max = np.max(all_static_concat, axis=0) if static_max is None else static_max

        # 归一化与样本切分 (包含时间连续性断层检测)
        for i in range(len(self.station_forcing)):
            self.station_forcing[i] = (self.station_forcing[i] - self.feat_min_f) / (self.feat_max_f - self.feat_min_f + 1e-8)
            self.station_state[i] = (self.station_state[i] - self.feat_min_s) / (self.feat_max_s - self.feat_min_s + 1e-8)
            self.station_static[i] = (self.station_static[i] - self.static_min) / (self.static_max - self.static_min + 1e-8)

            dates_array = self.station_dates[i]
            if len(dates_array) > 1:
                diffs = pd.Series(dates_array).diff()
                mode_step = diffs.mode()[0]
                expected_duration = mode_step * (self.seq_len - 1)

                num_samples = len(self.station_forcing[i]) - self.seq_len + 1
                if num_samples > 0:
                    for j in range(num_samples):
                        actual_duration = pd.Timedelta(dates_array[j + self.seq_len - 1] - dates_array[j])
                        if actual_duration == expected_duration:
                            self.samples.append((i, j))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        station_idx, start_idx = self.samples[idx]

        x_forcing = self.station_forcing[station_idx][start_idx : start_idx + self.seq_len]
        x_state = self.station_state[station_idx][start_idx : start_idx + self.seq_len]
        time_x = self.station_time_features[station_idx][start_idx : start_idx + self.seq_len]
        y = self.station_targets[station_idx][start_idx + self.seq_len - 1]
        target_date = self.station_dates[station_idx][start_idx + self.seq_len - 1]

        x_static = self.station_static[station_idx][start_idx : start_idx + self.seq_len]
        x_lc = self.station_lc[station_idx][start_idx : start_idx + self.seq_len]

        return (
            torch.tensor(x_forcing, dtype=torch.float32),
            torch.tensor(x_state, dtype=torch.float32),
            torch.tensor(time_x, dtype=torch.float32),
            torch.tensor(x_static, dtype=torch.float32),
            torch.tensor(x_lc, dtype=torch.long),
            torch.tensor(y, dtype=torch.float32),
            str(target_date)
        )

# ==========================================
# 2. TCN 基础模块定义
# ==========================================
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super(Chomp1d, self).__init__()
        self.chomp_size = chomp_size
    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout=0.2):
        super(TemporalBlock, self).__init__()
        self.conv1 = nn.Conv1d(n_inputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation)
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        self.conv2 = nn.Conv1d(n_outputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation)
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1, self.dropout1, self.conv2, self.chomp2, self.relu2, self.dropout2)
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()
    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TemporalConvNet(nn.Module):
    def __init__(self, num_inputs, num_channels, kernel_size=3, dropout=0.2):
        super(TemporalConvNet, self).__init__()
        layers = []
        num_levels = len(num_channels)
        for i in range(num_levels):
            dilation_size = 2 ** i
            in_channels = num_inputs if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers += [TemporalBlock(in_channels, out_channels, kernel_size, stride=1, dilation=dilation_size,
                                     padding=(kernel_size-1) * dilation_size, dropout=dropout)]
        self.network = nn.Sequential(*layers)
    def forward(self, x):
        return self.network(x)

# ==========================================
# 3. 交叉注意力融合模型
# ==========================================
class TCN_Transformer_CrossAttention(nn.Module):
    def __init__(self, num_forcing_features, num_state_features, seq_len,
                 num_static=2, num_lc_classes=10, lc_embed_dim=8,
                 d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.1):
        super(TCN_Transformer_CrossAttention, self).__init__()

        self.tcn = TemporalConvNet(num_inputs=num_forcing_features,
                                   num_channels=[d_model] * 6,
                                   kernel_size=3, dropout=dropout)

        self.lc_embedding = nn.Embedding(num_embeddings=num_lc_classes, embedding_dim=lc_embed_dim)

        combined_state_dim = num_state_features + num_static + lc_embed_dim
        self.state_linear = nn.Linear(combined_state_dim, d_model)
        self.time_projector = nn.Linear(4, d_model)

        encoder_layers = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
        self.cross_attention = nn.MultiheadAttention(embed_dim=d_model, num_heads=nhead, dropout=dropout, batch_first=True)

        self.regressor = nn.Sequential(
            nn.Linear(d_model, d_model // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(d_model // 2, d_model // 4), nn.ReLU(),
            nn.Linear(d_model // 4, 1)
        )

    def forward(self, x_forcing, x_state, time_x, x_static, x_lc):
        x_tcn_in = x_forcing.transpose(1, 2)
        f_tcn = self.tcn(x_tcn_in)
        f_met_memory = f_tcn.transpose(1, 2)

        lc_emb = self.lc_embedding(x_lc)
        combined_state = torch.cat([x_state, x_static, lc_emb], dim=-1)

        x_s_emb = self.state_linear(combined_state)
        time_emb = self.time_projector(time_x)
        x_state_combined = x_s_emb + time_emb
        f_state_global = self.transformer_encoder(x_state_combined)

        fused_features, _ = self.cross_attention(
            query=f_state_global,
            key=f_met_memory,
            value=f_met_memory
        )

        last_step_features = fused_features[:, -1, :]
        out = self.regressor(last_step_features)
        return out.squeeze(-1)

# ==========================================
# 4. SHAP 分析代码
# ==========================================
def perform_shap_analysis(model, dataloader, device, output_img_folder, forcing_cols, state_cols):
    print("\n🔍 开始执行 SHAP 变量重要性分析...")
    model.eval()

    shap_loader = DataLoader(dataloader.dataset, batch_size=4000, shuffle=True)
    batch = next(iter(shap_loader))

    batch_forcing, batch_state, batch_time = batch[0].to(device), batch[1].to(device), batch[2].to(device)
    batch_static, batch_lc = batch[3].to(device), batch[4].to(device)

    bg_size, test_size = 1000, 1000
    if batch_forcing.size(0) < (bg_size + test_size):
        print("⚠️ Batch size 太小，跳过 SHAP 分析。")
        return

    bg_f, bg_s = batch_forcing[:bg_size], batch_state[:bg_size]
    test_f, test_s = batch_forcing[bg_size:bg_size+test_size], batch_state[bg_size:bg_size+test_size]

    class SHAPWrapper(nn.Module):
        def __init__(self, base_model, t_ref, st_ref, lc_ref):
            super().__init__()
            self.base_model = base_model
            self.t_ref = t_ref[:1]
            self.st_ref = st_ref[:1]
            self.lc_ref = lc_ref[:1]

        def forward(self, x_f, x_s):
            b_size = x_f.size(0)
            t_x = self.t_ref.expand(b_size, -1, -1)
            x_st = self.st_ref.expand(b_size, -1, -1)
            x_l = self.lc_ref.expand(b_size, -1)
            out = self.base_model(x_f, x_s, t_x, x_st, x_l)
            return out.unsqueeze(-1)

    wrapper_model = SHAPWrapper(model, batch_time, batch_static, batch_lc).to(device)
    wrapper_model.eval()

    explainer = shap.GradientExplainer(wrapper_model, [bg_f, bg_s])
    shap_values = explainer.shap_values([test_f, test_s])

    if isinstance(shap_values, list) and len(shap_values) == 1 and isinstance(shap_values[0], list):
        shap_values = shap_values[0]

    shap_forcing = np.array(shap_values[0])
    shap_state = np.array(shap_values[1])

    shap_forcing_2d = shap_forcing.reshape(-1, len(forcing_cols))
    shap_state_2d = shap_state.reshape(-1, len(state_cols))

    test_f_2d = test_f.cpu().numpy().reshape(-1, len(forcing_cols))
    test_s_2d = test_s.cpu().numpy().reshape(-1, len(state_cols))

    shap_combined = np.concatenate([shap_forcing_2d, shap_state_2d], axis=1)
    features_combined = np.concatenate([test_f_2d, test_s_2d], axis=1)
    feature_names = forcing_cols + state_cols

    plt.figure(figsize=(12, 10))
    shap.summary_plot(shap_combined, features_combined, feature_names=feature_names, show=False)
    plt.title("SHAP Summary: Global Feature Impact on GPP (NT)", fontname='Arial', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(output_img_folder, "SHAP_Summary_Plot.png"), dpi=300)
    plt.close()

    plt.figure(figsize=(12, 10))
    shap.summary_plot(shap_combined, features_combined, feature_names=feature_names, plot_type="bar", show=False)
    plt.title("SHAP Global Feature Importance (Magnitude) for GPP (NT)", fontname='Arial', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(output_img_folder, "SHAP_Bar_Plot.png"), dpi=300)
    plt.close()

    print(f"✅ SHAP 生成完成，保存至: {output_img_folder}")

# ==========================================
# 5. 主流程
# ==========================================
def train_time_aware_model():
    seq_len = 96
    batch_size = 64
    num_epochs = 100
    learning_rate = 0.001
    patience = 10

    TIME_COLUMN_NAME = 'date'
    # 修改2: 显式定义目标列变量
    TARGET_COLUMN_NAME = 'GPP_NT_VUT_REF'
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    data_folder = r"C:\Users\Admin\Desktop\实验三\全波段全变量NT"
    # 建议顺便把输出文件夹名字里的DT改成NT，避免混淆
    output_img_folder = os.path.join(data_folder, "Results_TCN_Transformerkua分地类_NT")
    os.makedirs(output_img_folder, exist_ok=True)

    forcing_cols = ['SW_IN_F', 'SW_IN_POT', 'CO2_F_MDS', 'P_F', 'VPD_F', 'TA_F', 'TS_F_MDS_1', 'SWC_F_MDS_1', 'WS_F']
    state_cols = ['EPIC_Available_Mask', 'Band317nm_Ref', 'Band325nm_Ref', 'Band340nm_Ref',
                  'Band388nm_Ref', 'Band443nm_Ref', 'Band551nm_Ref', 'Band680nm_Ref',
                  'Band688nm_Ref', 'Band764nm_Ref', 'Band780nm_Ref', 'Mean_SZA', 'Mean_VZA', 'Mean_RAA']
    static_cols = ['Lat', 'Long']
    lc_col = 'Veg_ID'

    test_sites = ['US-xST_DBF', 'US-Bar_DBF', 'US-HB3_ENF', 'US-xNW_ENF', 'US-xKZ_GRA', 'US-xCP_GRA', 'US-A39_CRO', 'US-xJR_OSH', 'US-xTL_WET', 'US-SRM_WSA']
    val_sites = ['US-UMB_DBF', 'US-xBR_DBF', 'CA-TP1_ENF', 'US-xRM_ENF', 'US-Kon_GRA', 'US-xAE_GRA', 'US-Ne1_CRO', 'CA-Mer_WET', 'US-Whs_OSH', 'US-xDS_CVM']

    all_files = glob.glob(os.path.join(data_folder, "*.[cC][sS][vV]"))
    if not all_files:
        print("❌ 错误：未在指定目录找到CSV文件。")
        return

    train_files, val_files, test_files = [], [], []
    for f in all_files:
        fname = os.path.basename(f)
        if any(site in fname for site in test_sites):
            test_files.append(f)
        elif any(site in fname for site in val_sites):
            val_files.append(f)
        else:
            train_files.append(f)

    print(f"📁 共找到 {len(all_files)} 个站点文件。")
    print(f"   -> 训练集: {len(train_files)} | 验证集: {len(val_files)} | 测试集: {len(test_files)}")

    # 修改3: 在实例化 Dataset 时显式传入 target_col
    train_dataset = TimeAwareMultiStationDataset(
        train_files, seq_len, time_col=TIME_COLUMN_NAME, target_col=TARGET_COLUMN_NAME,
        forcing_cols=forcing_cols, state_cols=state_cols,
        static_cols=static_cols, lc_col=lc_col, split_type='train'
    )

    f_min_f, f_max_f = train_dataset.feat_min_f, train_dataset.feat_max_f
    f_min_s, f_max_s = train_dataset.feat_min_s, train_dataset.feat_max_s
    s_min, s_max = train_dataset.static_min, train_dataset.static_max

    val_dataset = TimeAwareMultiStationDataset(
        val_files, seq_len, time_col=TIME_COLUMN_NAME, target_col=TARGET_COLUMN_NAME,
        forcing_cols=forcing_cols, state_cols=state_cols,
        static_cols=static_cols, lc_col=lc_col,
        feat_min_f=f_min_f, feat_max_f=f_max_f, feat_min_s=f_min_s, feat_max_s=f_max_s,
        static_min=s_min, static_max=s_max, split_type='val'
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    model = TCN_Transformer_CrossAttention(
        num_forcing_features=len(forcing_cols),
        num_state_features=len(state_cols),
        seq_len=seq_len,
        num_static=len(static_cols),
        num_lc_classes=10,
        lc_embed_dim=8
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    checkpoint_latest_path = os.path.join(output_img_folder, "checkpoint_latest.pth")
    checkpoint_best_path = os.path.join(output_img_folder, "checkpoint_best.pth")

    start_epoch = 0
    best_val_loss = float('inf')
    epochs_no_improve = 0

    if os.path.exists(checkpoint_latest_path):
        print(f"\n🔄 检测到本地进度，尝试恢复...")
        try:
            checkpoint = torch.load(checkpoint_latest_path, map_location=device)
            model.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            start_epoch = checkpoint['epoch'] + 1
            best_val_loss = checkpoint['best_val_loss']
            epochs_no_improve = checkpoint['epochs_no_improve']
            print(f"✅ 从第 {start_epoch} 轮恢复，历史最佳 MSE: {best_val_loss:.4f}")
        except Exception as e:
            print(f"⚠️ 无法加载历史进度，将重新开始。错误信息: {e}")

    if start_epoch < num_epochs and epochs_no_improve < patience:
        print(f"🚀 开始训练...\n" + "-"*40)
        for epoch in range(start_epoch, num_epochs):
            model.train()
            train_loss = 0

            for batch_forcing, batch_state, batch_time, batch_static, batch_lc, batch_y, _ in train_loader:
                batch_forcing, batch_state, batch_time = batch_forcing.to(device), batch_state.to(device), batch_time.to(device)
                batch_static, batch_lc, batch_y = batch_static.to(device), batch_lc.to(device), batch_y.to(device)

                optimizer.zero_grad()
                outputs = model(batch_forcing, batch_state, batch_time, batch_static, batch_lc)
                loss = criterion(outputs, batch_y)
                loss.backward()

                # 梯度裁剪
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

                optimizer.step()
                train_loss += loss.item()

            model.eval()
            val_loss = 0
            with torch.no_grad():
                for batch_forcing, batch_state, batch_time, batch_static, batch_lc, batch_y, _ in val_loader:
                    batch_forcing, batch_state, batch_time = batch_forcing.to(device), batch_state.to(device), batch_time.to(device)
                    batch_static, batch_lc, batch_y = batch_static.to(device), batch_lc.to(device), batch_y.to(device)

                    outputs = model(batch_forcing, batch_state, batch_time, batch_static, batch_lc)
                    loss = criterion(outputs, batch_y)
                    val_loss += loss.item()

            avg_train_loss = train_loss / len(train_loader)
            avg_val_loss = val_loss / len(val_loader)
            print(f"Epoch [{epoch+1:03d}/{num_epochs}] | Train MSE: {avg_train_loss:.4f} | Val MSE: {avg_val_loss:.4f}")

            if avg_val_loss < best_val_loss:
                print(f"   🌟 新的最佳模型！MSE: {avg_val_loss:.4f}。")
                best_val_loss = avg_val_loss
                epochs_no_improve = 0
                torch.save(model.state_dict(), checkpoint_best_path)
            else:
                epochs_no_improve += 1
                print(f"   ⏳ 验证集未改善 (早停: {epochs_no_improve}/{patience})")

            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_loss': best_val_loss,
                'epochs_no_improve': epochs_no_improve
            }, checkpoint_latest_path)

            if epochs_no_improve >= patience:
                print(f"\n🛑 早停机制触发，训练结束。")
                break
            print("-" * 40)

   # 6. 分站点独立测试评估 (整合高级绘图逻辑)
    # ==========================================
    if os.path.exists(checkpoint_best_path):
        print(f"\n🎯 开始测试集评估...")
        model.load_state_dict(torch.load(checkpoint_best_path, map_location=device))
    model.eval()

    global_all_preds, global_all_targets = [], []
    for test_file in test_files:
        station_name = os.path.basename(test_file).replace('.csv', '')
        single_test_dataset = TimeAwareMultiStationDataset(
            [test_file], seq_len, time_col=TIME_COLUMN_NAME,
            forcing_cols=forcing_cols, state_cols=state_cols,
            static_cols=static_cols, lc_col=lc_col,
            feat_min_f=f_min_f, feat_max_f=f_max_f, feat_min_s=f_min_s, feat_max_s=f_max_s,
            static_min=s_min, static_max=s_max, split_type='test'
        )

        if len(single_test_dataset) == 0: continue

        single_test_loader = DataLoader(single_test_dataset, batch_size=batch_size, shuffle=False)
        all_preds, all_targets, all_times = [], [], []

        with torch.no_grad():
            for batch_forcing, batch_state, batch_time, batch_static, batch_lc, batch_y, batch_dt in single_test_loader:
                batch_forcing, batch_state, batch_time = batch_forcing.to(device), batch_state.to(device), batch_time.to(device)
                batch_static, batch_lc = batch_static.to(device), batch_lc.to(device)

                outputs = model(batch_forcing, batch_state, batch_time, batch_static, batch_lc)
                all_preds.extend(outputs.cpu().numpy())
                all_targets.extend(batch_y.numpy())
                all_times.extend(batch_dt)

        all_preds, all_targets = np.array(all_preds), np.array(all_targets)
        all_times = pd.to_datetime(all_times)


        # 放宽数据清洗要求：允许夜间正常波动的轻微负值，保留 GPP > -5 的数据
        valid_mask = all_targets > -5
        plot_targets = all_targets[valid_mask]
        plot_preds = all_preds[valid_mask]
        plot_times = all_times[valid_mask]

        if len(plot_targets) == 0:
            continue

        global_all_preds.extend(plot_preds)
        global_all_targets.extend(plot_targets)

        station_mse = np.mean((plot_preds - plot_targets)**2)
        station_r2 = r2_score(plot_targets, plot_preds)

        print(f"📢 站点: {station_name} | 测试集 MSE: {station_mse:.4f} | 测试集 R²: {station_r2:.4f}")

        # 图表 1: 滑动平均趋势图 (保持不变，仅修复黑底)
        window_size = 12
        all_targets_smooth = pd.Series(plot_targets).rolling(window=window_size, min_periods=1).mean().values
        all_preds_smooth = pd.Series(plot_preds).rolling(window=window_size, min_periods=1).mean().values

        plt.figure(figsize=(15, 6))
        plt.plot(plot_times, plot_targets, color='royalblue', linewidth=0.5, alpha=0.2, label='Actual (Raw)')
        plt.plot(plot_times, plot_preds, color='crimson', linewidth=0.5, alpha=0.2, label='Predicted (Raw)')
        plt.plot(plot_times, all_targets_smooth, label=f'Actual GPP (MA-{window_size})', color='royalblue', linewidth=1.5, alpha=0.9)
        plt.plot(plot_times, all_preds_smooth, label=f'Predicted GPP (MA-{window_size})', color='crimson', linewidth=1.5, linestyle='--', alpha=0.9)

        plt.title(f'[{station_name}] Test Set GPP Prediction (Moving Average Window={window_size})', fontsize=14, fontname='Arial')
        plt.xlabel('Time', fontsize=12, fontname='Arial')
        plt.ylabel('GPP Value', fontsize=12, fontname='Arial')
        plt.legend(prop={'family': 'Arial'})

        ax = plt.gca()
        ax.tick_params(direction='in')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        plt.grid(True, which='both', linestyle='--', alpha=0.5)
        plt.tight_layout()
        # 强制指定白色背景，关闭透明度
        plt.savefig(os.path.join(output_img_folder, f"{station_name}_trend_ma.png"), dpi=300, facecolor='white', transparent=False)
        plt.close()

        # 图表 2: 局部放大图 (30天) —— 修改为使用 Raw Data
        zoom_days = 30
        steps_per_day = 24
        zoom_steps = zoom_days * steps_per_day
        zoom_steps = min(zoom_steps, len(plot_times))

        if zoom_steps > 0:
            peak_idx = np.argmax(all_targets_smooth)
            start_idx = max(0, peak_idx - zoom_steps // 2)
            end_idx = min(len(plot_times), start_idx + zoom_steps)

            if end_idx - start_idx < zoom_steps:
                start_idx = max(0, end_idx - zoom_steps)

            plt.figure(figsize=(15, 5))
            # 修改：此处换为 plot_targets 和 plot_preds（原始序列），以展示细节和高频变化
            plt.plot(plot_times[start_idx:end_idx], plot_targets[start_idx:end_idx],
                     label='Actual GPP (Raw)', color='royalblue', linewidth=1.5)
            plt.plot(plot_times[start_idx:end_idx], plot_preds[start_idx:end_idx],
                     label='Predicted GPP (Raw)', color='crimson', linewidth=1.5, linestyle='--', alpha=0.8)

            peak_date_str = pd.to_datetime(plot_times.iloc[peak_idx] if isinstance(plot_times, pd.Series) else plot_times[peak_idx]).strftime('%Y-%m-%d')

            plt.title(f'[{station_name}] 30-Day Zoomed Prediction (Raw Data Detail around {peak_date_str})', fontsize=14, fontname='Arial')
            plt.xlabel('Time', fontsize=12, fontname='Arial')
            plt.ylabel('GPP Value', fontsize=12, fontname='Arial')
            plt.legend(prop={'family': 'Arial'})

            ax = plt.gca()
            ax.tick_params(direction='in')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            plt.grid(True, which='both', linestyle='--', alpha=0.6)
            plt.xticks(rotation=30)
            plt.tight_layout()
            plt.savefig(os.path.join(output_img_folder, f"{station_name}_zoom_30days.png"), dpi=300, facecolor='white', transparent=False)
            plt.close()

        # 图表 3: 散点图
        plt.figure(figsize=(6, 6))
        plt.scatter(plot_targets, plot_preds, alpha=0.6, color='teal', s=15, edgecolors='k', linewidth=0.2)

        min_val = min(plot_targets.min(), plot_preds.min())
        max_val = max(plot_targets.max(), plot_preds.max())
        plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='1:1 Line')

        plt.title(f'[{station_name}] Actual vs Predicted Scatter', fontname='Arial')
        plt.xlabel('Actual GPP', fontname='Arial')
        plt.ylabel('Predicted GPP', fontname='Arial')
        plt.legend(prop={'family': 'Arial'})

        ax = plt.gca()
        ax.tick_params(direction='in')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        plt.tight_layout()
        plt.savefig(os.path.join(output_img_folder, f"{station_name}_scatter.png"), dpi=300, facecolor='white', transparent=False)
        plt.close()

        # ==========================================
        # 新增: 图表 4: 抽取任一年的数据展示时间序列
        # ==========================================
        years = plot_times.dt.year if hasattr(plot_times, 'dt') else pd.Series(plot_times).dt.year
        unique_years = years.unique()

        if len(unique_years) > 0:
            # 智能选择策略：为了图表丰满，默认选择数据点最多的一年，避免选到跨年只剩几天的数据
            target_year = years.value_counts().idxmax()
            year_mask = (years == target_year).values

            y_times = plot_times[year_mask]
            y_targets = plot_targets[year_mask]
            y_preds = plot_preds[year_mask]

            plt.figure(figsize=(15, 5))
            plt.plot(y_times, y_targets, label=f'Actual GPP ({target_year})', color='royalblue', linewidth=1.2, alpha=0.8)
            plt.plot(y_times, y_preds, label=f'Predicted GPP ({target_year})', color='crimson', linewidth=1.2, linestyle='--', alpha=0.8)

            plt.title(f'[{station_name}] Full Year GPP Time Series ({target_year})', fontsize=14, fontname='Arial')
            plt.xlabel('Time', fontsize=12, fontname='Arial')
            plt.ylabel('GPP Value', fontsize=12, fontname='Arial')
            plt.legend(prop={'family': 'Arial'}, loc='upper right')

            ax = plt.gca()
            ax.tick_params(direction='in')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            plt.grid(True, which='both', linestyle='--', alpha=0.4)
            plt.tight_layout()
            plt.savefig(os.path.join(output_img_folder, f"{station_name}_singleyear_{target_year}.png"), dpi=300, facecolor='white', transparent=False)
            plt.close()

    # ------------------------------------------
    # 7. 全局指标汇总与 SHAP 分析
    # ------------------------------------------
    if len(global_all_targets) > 0:
        global_all_preds = np.array(global_all_preds)
        global_all_targets = np.array(global_all_targets)

        global_mse = np.mean((global_all_preds - global_all_targets)**2)
        global_r2 = r2_score(global_all_targets, global_all_preds)

        print("\n" + "="*50)
        print("🌎 所有站点测试集全局评估结果")
        print("="*50)
        print(f"有效总测试样本数 (剔除 GPP<-5): {len(global_all_targets)}")
        print(f"Global Test MSE (NT): {global_mse:.4f}")
        print(f"Global Test R² Score (NT): {global_r2:.4f}")
        print("="*50)

        try:
            perform_shap_analysis(model, val_loader, device, output_img_folder, forcing_cols, state_cols)
        except Exception as e:
            pass

if __name__ == "__main__":
    train_time_aware_model()
#夜间GPP

D:\miniconda3\envs\dl_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📁 共找到 77 个站点文件。
   -> 训练集: 57 | 验证集: 10 | 测试集: 10

🔄 检测到本地进度，尝试恢复...
✅ 从第 38 轮恢复，历史最佳 MSE: 19.6395
🚀 开始训练...
----------------------------------------
Epoch [039/100] | Train MSE: 5.1717 | Val MSE: 20.8186
   ⏳ 验证集未改善 (早停: 2/10)
----------------------------------------
Epoch [040/100] | Train MSE: 5.1792 | Val MSE: 21.2153
   ⏳ 验证集未改善 (早停: 3/10)
----------------------------------------
Epoch [041/100] | Train MSE: 5.1543 | Val MSE: 22.8984
   ⏳ 验证集未改善 (早停: 4/10)
----------------------------------------
Epoch [042/100] | Train MSE: 5.1208 | Val MSE: 21.5246
   ⏳ 验证集未改善 (早停: 5/10)
----------------------------------------
Epoch [043/100] | Train MSE: 5.1564 | Val MSE: 20.5096
   ⏳ 验证集未改善 (早停: 6/10)
----------------------------------------
Epoch [044/100] | Train MSE: 5.1026 | Val MSE: 21.3744
   ⏳ 验证集未改善 (早停: 7/10)
----------------------------------------
Epoch [045/100] | Train MSE: 5.1018 | Val MSE: 20.5067
   ⏳ 验证集未改善 (早停: 8/10)
----------------------------------------
Epoch [046/100] |

C:\Users\admin\AppData\Local\Temp\ipykernel_24480\802157284.py:297: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_combined, features_combined, feature_names=feature_names, show=False)
C:\Users\admin\AppData\Local\Temp\ipykernel_24480\802157284.py:304: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(shap_combined, features_combined, feature_names=feature_names, plot_type="bar", show=False)


✅ SHAP 生成完成，保存至: C:\Users\Admin\Desktop\实验三\全波段全变量NT\Results_TCN_Transformerkua分地类_NT


In [7]:
import os
import glob
from typing import Iterable, Union, List, Dict

import numpy as np
import pandas as pd


# ============================================================
# 用户参数区
# ============================================================
INPUT_PATH = r"C:\Users\admin\Desktop\ERA5_land_extracted_data.csv"      # 可以是单个 CSV，也可以是文件夹
OUTPUT_CSV = r"c:\Users\admin\Desktop\FR-Fon ERA5Land_FLUXNET_HR.csv"

TIME_COL = "valid_time"                # ERA5-Land 时间列
LOCAL_UTC_OFFSET_HOURS =0             # 如果要北京时间/东八区，改成 8
MISSING_VALUE = -9999

# ERA5-Land CDS 小时数据的 ssrd/strd/ssr/str/tp 通常是“从当天 00 UTC 起累积到 valid_time”的量。
# 如果你的输入已经提前拆成逐小时增量，把它改成 "already_hourly"。
ACCUMULATION_MODE = "era5land_cumulative"


def _expand_csv_paths(input_path: Union[str, Iterable[str]]) -> List[str]:
    if isinstance(input_path, (list, tuple, set)):
        paths = []
        for p in input_path:
            paths.extend(_expand_csv_paths(p))
        return sorted(set(paths))

    input_path = os.fspath(input_path)

    if os.path.isdir(input_path):
        return sorted(glob.glob(os.path.join(input_path, "*.csv")))

    paths = sorted(glob.glob(input_path))
    if not paths and os.path.isfile(input_path):
        paths = [input_path]

    if not paths:
        raise FileNotFoundError(f"没有找到 CSV 文件: {input_path}")

    return paths


def read_and_merge_era5land_csvs(input_path, time_col="valid_time") -> pd.DataFrame:
    """
    读取一个或多个 ERA5-Land CSV，并按时间合并。
    如果多个文件包含相同时间、不同变量，会自动合并到同一行。
    """
    paths = _expand_csv_paths(input_path)
    frames = []

    for path in paths:
        df = pd.read_csv(path)

        if time_col not in df.columns:
            raise ValueError(f"{os.path.basename(path)} 中找不到时间列 {time_col}")

        df[time_col] = pd.to_datetime(df[time_col], utc=True, errors="coerce").dt.tz_convert(None)

        if df[time_col].isna().any():
            bad = df[df[time_col].isna()].head()
            raise ValueError(f"{os.path.basename(path)} 中存在无法解析的时间，例如：\n{bad}")

        df = df.set_index(time_col).sort_index()

        for c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

        frames.append(df)

    merged = pd.concat(frames, axis=0, sort=True)

    # 同一时间重复行取均值；不同文件不同变量会自然合并
    merged = merged.groupby(level=0).mean(numeric_only=True)
    merged = merged.sort_index()

    # 去掉所有 ERA5 变量都为空的边界行
    merged = merged.dropna(how="all")

    # 补齐逐小时时间轴
    full_index = pd.date_range(merged.index.min(), merged.index.max(), freq="h")
    merged = merged.reindex(full_index)
    merged.index.name = time_col

    return merged


def deaccumulate_era5land(series: pd.Series,
                          nonnegative: bool = True,
                          small_negative_tol: float = 1e-8) -> pd.Series:
    """
    ERA5-Land 累积量 -> 每小时增量。

    valid_time 01:00：表示 00:00-01:00，直接取原值；
    valid_time 02:00-23:00：用当前累积量 - 前一小时累积量；
    valid_time 00:00：通常是前一天 step=24，需要与前一天 23:00 相减。
    """
    s = pd.to_numeric(series, errors="coerce").astype(float)
    idx = s.index

    prev = s.shift(1)
    prev_idx = pd.Series(idx, index=idx).shift(1)
    contiguous = (pd.Series(idx, index=idx) - prev_idx) == pd.Timedelta(hours=1)

    hour = pd.Series(idx.hour, index=idx)
    out = pd.Series(np.nan, index=idx, dtype=float)

    mask_h01 = hour.eq(1)
    out.loc[mask_h01] = s.loc[mask_h01]

    mask_diff = (~mask_h01) & contiguous
    out.loc[mask_diff] = s.loc[mask_diff] - prev.loc[mask_diff]

    if nonnegative:
        tiny_neg = out.between(-small_negative_tol, 0)
        out.loc[tiny_neg] = 0.0
        out.loc[out < 0] = np.nan

    return out


def saturation_vapor_pressure_hpa(temp_c: pd.Series) -> pd.Series:
    """Magnus 公式，返回 hPa。"""
    return 6.112 * np.exp((17.67 * temp_c) / (temp_c + 243.5))


def format_fluxnet_timestamp(dt_index: pd.DatetimeIndex) -> pd.Series:
    """FLUXNET 常用短格式：YYYYMMDDHHMM。"""
    return pd.Series(dt_index.strftime("%Y%m%d%H%M"), index=dt_index)


def _col(df: pd.DataFrame, name: str) -> pd.Series:
    if name in df.columns:
        return df[name]
    return pd.Series(np.nan, index=df.index, dtype=float)


def build_fluxnet_hourly(era: pd.DataFrame,
                         local_utc_offset_hours: int = 0,
                         accumulation_mode: str = "era5land_cumulative",
                         missing_value: float = -9999) -> pd.DataFrame:

    df = era.copy().sort_index()

    if accumulation_mode not in {"era5land_cumulative", "already_hourly"}:
        raise ValueError("accumulation_mode 只能是 'era5land_cumulative' 或 'already_hourly'")

    # ========================================================
    # 1. ERA5-Land 累积量去累积
    # ========================================================
    acc: Dict[str, pd.Series] = {}

    accumulated_nonnegative = ["ssrd", "strd", "ssr", "tp", "tisr"]
    accumulated_signed = ["str"]

    for c in accumulated_nonnegative:
        if c in df.columns:
            if accumulation_mode == "era5land_cumulative":
                # 辐射 GRIB 打包可能有几个 J m-2 的微小负差分
                tol = 50.0 if c in {"ssrd", "strd", "ssr", "tisr"} else 1e-6
                acc[c] = deaccumulate_era5land(df[c], nonnegative=True, small_negative_tol=tol)
            else:
                acc[c] = df[c].astype(float)

    for c in accumulated_signed:
        if c in df.columns:
            if accumulation_mode == "era5land_cumulative":
                acc[c] = deaccumulate_era5land(df[c], nonnegative=False)
            else:
                acc[c] = df[c].astype(float)

    # ========================================================
    # 2. 时间列
    # ========================================================
    time_end_local = df.index + pd.Timedelta(hours=local_utc_offset_hours)
    time_start_local = time_end_local - pd.Timedelta(hours=1)

    out = pd.DataFrame(index=df.index)
    out["TIMESTAMP_START"] = format_fluxnet_timestamp(time_start_local).values
    out["TIMESTAMP_END"] = format_fluxnet_timestamp(time_end_local).values

    # ========================================================
    # 3. 瞬时变量 / 状态变量
    # ========================================================
    t2m_c = _col(df, "t2m") - 273.15
    d2m_c = _col(df, "d2m") - 273.15

    out["TA_F"] = t2m_c
    out["TS_F_MDS_1"] = _col(df, "stl1") - 273.15
    out["SWC_F_MDS_1"] = _col(df, "swvl1") * 100.0  # 乘以100，将小数(如0.33)转换为百分比(33.0)
    out["PA_F"] = _col(df, "sp") / 1000.0
    out["WS_F"] = np.sqrt(_col(df, "u10") ** 2 + _col(df, "v10") ** 2)

    es = saturation_vapor_pressure_hpa(t2m_c)
    ea = saturation_vapor_pressure_hpa(d2m_c)

    out["VPD_F"] = (es - ea).clip(lower=0)
    out["RH"] = (100.0 * ea / es).clip(lower=0, upper=100)

    # ========================================================
    # 4. 辐射和降水
    # ========================================================
    if "ssrd" in acc:
        out["SW_IN_F"] = acc["ssrd"] / 3600.0

    if "ssrd" in acc and "ssr" in acc:
        # ssr = SW_down - SW_up
        out["SW_OUT"] = (acc["ssrd"] - acc["ssr"]) / 3600.0

    if "strd" in acc:
        out["LW_IN_F"] = acc["strd"] / 3600.0

    if "strd" in acc and "str" in acc:
        # str = LW_down - LW_up，通常为负
        out["LW_OUT"] = (acc["strd"] - acc["str"]) / 3600.0

    if "tp" in acc:
        # m -> mm
        out["P_F"] = acc["tp"] * 1000.0

    if "tisr" in acc:
        out["SW_IN_POT"] = acc["tisr"] / 3600.0

    if {"SW_IN_F", "SW_OUT", "LW_IN_F", "LW_OUT"}.issubset(out.columns):
        out["NETRAD"] = out["SW_IN_F"] - out["SW_OUT"] + out["LW_IN_F"] - out["LW_OUT"]

    # ========================================================
    # 5. 合理性清理
    # ========================================================
    nonnegative_out = [
        "SW_IN_F", "SW_OUT", "LW_IN_F", "LW_OUT",
        "P_F", "SW_IN_POT", "WS_F", "VPD_F", "RH"
    ]

    for c in nonnegative_out:
        if c in out.columns:
            tol = 0.05 if c in {"SW_IN_F", "SW_OUT", "LW_IN_F", "LW_OUT", "SW_IN_POT"} else 1e-6
            tiny_neg = out[c].between(-tol, 0)
            out.loc[tiny_neg, c] = 0.0
            out.loc[out[c] < 0, c] = np.nan

    preferred_order = [
        "TIMESTAMP_START", "TIMESTAMP_END",
        "TA_F", "VPD_F", "RH", "PA_F", "WS_F",
        "SW_IN_F", "SW_OUT", "LW_IN_F", "LW_OUT", "NETRAD",
        "P_F", "TS_F_MDS_1", "SWC_F_MDS_1", "SW_IN_POT"
    ]

    cols = [c for c in preferred_order if c in out.columns]
    out = out[cols]

    numeric_cols = [c for c in out.columns if c not in ["TIMESTAMP_START", "TIMESTAMP_END"]]
    out[numeric_cols] = out[numeric_cols].replace([np.inf, -np.inf], np.nan).round(6)
    out[numeric_cols] = out[numeric_cols].fillna(missing_value)

    return out.reset_index(drop=True)


def convert_era5land_to_fluxnet(input_path,
                                output_csv,
                                time_col="valid_time",
                                local_utc_offset_hours=0,
                                accumulation_mode="era5land_cumulative",
                                missing_value=-9999) -> pd.DataFrame:

    era = read_and_merge_era5land_csvs(input_path=input_path, time_col=time_col)

    flux = build_fluxnet_hourly(
        era,
        local_utc_offset_hours=local_utc_offset_hours,
        accumulation_mode=accumulation_mode,
        missing_value=missing_value
    )

    flux.to_csv(output_csv, index=False)

    print(f"完成：{output_csv}")
    print(f"输出行数: {len(flux)}")
    print(f"输出列: {list(flux.columns)}")

    return flux


if __name__ == "__main__":
    convert_era5land_to_fluxnet(
        input_path=INPUT_PATH,
        output_csv=OUTPUT_CSV,
        time_col=TIME_COL,
        local_utc_offset_hours=LOCAL_UTC_OFFSET_HOURS,
        accumulation_mode=ACCUMULATION_MODE,
        missing_value=MISSING_VALUE
    )

完成：c:\Users\admin\Desktop\FR-Fon ERA5Land_FLUXNET_HR.csv
输出行数: 744
输出列: ['TIMESTAMP_START', 'TIMESTAMP_END', 'TA_F', 'VPD_F', 'RH', 'PA_F', 'WS_F', 'SW_IN_F', 'SW_OUT', 'LW_IN_F', 'LW_OUT', 'NETRAD', 'P_F', 'TS_F_MDS_1', 'SWC_F_MDS_1']
